In [1]:
!pip install yfinance openpyxl plotly --quiet

# Módulo 3: Rebalanceo Óptimo (Programación Dinámica)

Este notebook implementa el rebalanceo óptimo de portafolios considerando costos de transacción. Se utiliza la ecuación de Bellman mediante el método de inducción hacia atrás (Backward Induction) para construir la matriz de costos y definir la política óptima de rebalanceo. Luego se realiza una simulación histórica hacia adelante (Forward Simulation) para comparar la estrategia optimizada por programación dinámica con las de Buy & Hold y Siempre Rebalancear.

In [2]:
import os
import json
import io
import time
import itertools
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from scipy.spatial.distance import cdist

# Crear carpeta de salidas si no existe
os.makedirs("salidas", exist_ok=True)

# Parámetros globales
TICKERS = ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']
FECHA_INICIO = '2015-01-01'
FECHA_FIN = '2024-12-31'
CAPITAL_INICIAL = 100000
LAMBDA_TC = 0.0010
T_PERIODOS = 12
PASO_GRILLA = 0.10

## 1. Carga de Insumos y Descarga de Datos

Intentamos cargar los pesos óptimos del portafolio generados en el Módulo 1 desde `resultados_m1.json` (o usamos equiponderados en su defecto) y descargamos los precios históricos de Yahoo Finance, aplicando el remuestreo a frecuencia mensual.

In [3]:
# 1. Cargar pesos óptimos de Módulo 1
w_optimo = np.ones(len(TICKERS)) / len(TICKERS)
pesos_dict = {}
if os.path.exists("resultados_m1.json"):
    with open("resultados_m1.json", "r", encoding="utf-8") as f:
        datos_m1 = json.load(f)
        if "markowitz_max_sharpe" in datos_m1:
            pesos_dict = datos_m1["markowitz_max_sharpe"]["pesos"]
            w_optimo = np.array([pesos_dict.get(t, 1/len(TICKERS)) for t in TICKERS], dtype=float)
            if w_optimo.sum() > 0:
                w_optimo /= w_optimo.sum()

print("Pesos óptimos objetivo (Markowitz):", dict(zip(TICKERS, w_optimo.round(4))))

# 2. Descargar precios
print(f"Descargando datos históricos...")
descarga = yf.download(TICKERS, start=FECHA_INICIO, end=FECHA_FIN, progress=False)
if isinstance(descarga, pd.DataFrame) and not descarga.empty:
    precios = descarga['Close'] if 'Close' in descarga.columns else descarga
else:
    raise ValueError("Error al descargar datos de Yahoo Finance.")

precios = precios.ffill().bfill()
TICKERS_ORDENADOS = list(precios.columns)

# Recalcular pesos óptimos alineados a las columnas reales descargadas
w_optimo = np.array([pesos_dict.get(t, 1/len(TICKERS_ORDENADOS)) for t in TICKERS_ORDENADOS], dtype=float)
w_optimo /= w_optimo.sum()

# Resampleo mensual
frecuencia_sel = "Mensual"
codigo_freq = "ME"
precios_resampleados = precios.resample(codigo_freq).last()
retornos_resampleados = precios_resampleados.pct_change().dropna()

# Tomar exactamente los últimos T_PERIODOS para la simulación DP
if len(retornos_resampleados) > T_PERIODOS:
    retornos_sim = retornos_resampleados.iloc[-T_PERIODOS:]
else:
    retornos_sim = retornos_resampleados
    T_PERIODOS = len(retornos_sim)

fechas_sim = retornos_sim.index

# Retornos diarios, mu y Sigma para gráficos descriptivos
retornos_diarios = pd.DataFrame(np.log(precios / precios.shift(1))).dropna()
mu = retornos_diarios.mean().to_numpy(dtype=float) * 252
Sigma = retornos_diarios.cov().to_numpy(dtype=float) * 252

print(f"Datos cargados con éxito. Periodos de rebalanceo a simular: {T_PERIODOS} meses")

Pesos óptimos objetivo (Markowitz): {'ABX.TO': np.float64(0.3773), 'BHP': np.float64(0.6227), 'BVN': np.float64(0.0), 'FSM': np.float64(0.0), 'VOLCABC1.LM': np.float64(0.0)}
Descargando datos históricos...


Datos cargados con éxito. Periodos de rebalanceo a simular: 12 meses


## 2. Creación del Espacio de Estados (Grilla)

Generamos la grilla discreta de combinaciones posibles de pesos que sumen exactamente 1.0 utilizando el salto de discretización especificado ($0.10$).

In [4]:
def generar_grilla(n, paso):
    valores = np.arange(0, 1 + paso, paso)
    grilla = [p for p in itertools.product(valores, repeat=n) if np.isclose(sum(p), 1.0)]
    return np.array(grilla, dtype=float)

S = generar_grilla(len(TICKERS_ORDENADOS), PASO_GRILLA)
M = len(S)
print(f"Número total de estados válidos generados en la grilla (M): {M}")

Número total de estados válidos generados en la grilla (M): 1001


## 3. Inducción Hacia Atrás (Ecuación de Bellman)

Implementamos el algoritmo de inducción hacia atrás para calcular la matriz de costo acumulado $V$ y determinar la política óptima para cada periodo y estado de la grilla.

In [5]:
print("Calculando distancias y costos inmediatos...")
# Matriz de costos de transacción usando la distancia de Manhattan (L1 / cityblock)
TC_matrix = LAMBDA_TC * cdist(S, S, metric='cityblock')
# Costo inmediato por suboptimalidad (desviación cuadrática del óptimo de Markowitz)
Subopt_cost = np.sum((S - w_optimo)**2, axis=1)

# Inicializar Bellman
V = np.zeros((T_PERIODOS + 1, M))
politica = np.zeros((T_PERIODOS, M), dtype=int)

print("Resolviendo recursión de Bellman...")
for t in range(T_PERIODOS - 1, -1, -1):
    costo_j = Subopt_cost + V[t+1]
    Cost_matrix = TC_matrix + costo_j
    mejor_accion = np.argmin(Cost_matrix, axis=1)
    V[t] = Cost_matrix[np.arange(M), mejor_accion]
    politica[t] = mejor_accion

print("Matriz de costos de Bellman calculada exitosamente.")

Calculando distancias y costos inmediatos...
Resolviendo recursión de Bellman...
Matriz de costos de Bellman calculada exitosamente.


## 4. Simulación de Riqueza Histórica (Forward Simulation)

Corremos la simulación hacia adelante sobre la trayectoria histórica completa y calculamos la riqueza acumulada y caídas máximas (drawdowns) para las tres estrategias (DP, Siempre Rebalancear y Buy & Hold).

In [6]:
# Obtenemos la trayectoria completa para graficar
precios_res_full = precios.resample(codigo_freq).last()
retornos_sim_full = precios_res_full.pct_change().dropna()
T_full = len(retornos_sim_full)
fechas_sim_full = [precios_res_full.index[0]] + list(retornos_sim_full.index)

W_bh = np.zeros(T_full + 1)
W_sr = np.zeros(T_full + 1)
W_dp = np.zeros(T_full + 1)

W_bh[0] = W_sr[0] = W_dp[0] = CAPITAL_INICIAL

w_bh = w_optimo.copy()
w_sr = w_optimo.copy()
w_dp = w_optimo.copy()

rebalanceos_dp = []

for t in range(T_full):
    ret = np.array(retornos_sim_full.iloc[t].values, dtype=float)
    
    # 1. Buy & Hold
    ret_bh = np.dot(w_bh, ret)
    W_bh[t+1] = W_bh[t] * (1 + ret_bh)
    w_bh = w_bh * (1 + ret) / (1 + ret_bh)
    
    # 2. Siempre Rebalancear
    if t > 0:
        tc_sr = LAMBDA_TC * np.sum(np.abs(w_sr - w_optimo))
        capital_post_tc_sr = W_sr[t] * (1 - tc_sr)
        ret_sr = np.dot(w_optimo, ret)
        W_sr[t+1] = capital_post_tc_sr * (1 + ret_sr)
        w_sr = w_optimo * (1 + ret) / (1 + ret_sr)
    else:
        ret_sr = np.dot(w_sr, ret)
        W_sr[t+1] = W_sr[t] * (1 + ret_sr)
        w_sr = w_sr * (1 + ret) / (1 + ret_sr)
    
    # 3. Programación Dinámica
    if t > 0:
        idx_s = int(np.argmin(np.sum(np.abs(S - w_dp), axis=1)))
        idx_a = int(politica[min(t, T_PERIODOS-1), idx_s])
        w_dp_nuevo = S[idx_a]
        
        if idx_s != idx_a:
            rebalanceos_dp.append(t)
            tc_dp = LAMBDA_TC * np.sum(np.abs(w_dp - w_dp_nuevo))
            capital_post_tc_dp = W_dp[t] * (1 - tc_dp)
            ret_dp = np.dot(w_dp_nuevo, ret)
            W_dp[t+1] = capital_post_tc_dp * (1 + ret_dp)
            w_dp = w_dp_nuevo * (1 + ret) / (1 + ret_dp)
        else:
            ret_dp = np.dot(w_dp, ret)
            W_dp[t+1] = W_dp[t] * (1 + ret_dp)
            w_dp = w_dp * (1 + ret) / (1 + ret_dp)
    else:
        ret_dp = np.dot(w_dp, ret)
        W_dp[t+1] = W_dp[t] * (1 + ret_dp)
        w_dp = w_dp * (1 + ret) / (1 + ret_dp)

print("Simulación histórica completada.")
print(f"  Riqueza Final DP: ${W_dp[-1]:,.2f}")
print(f"  Riqueza Final Siempre Rebalancear: ${W_sr[-1]:,.2f}")
print(f"  Riqueza Final Buy & Hold: ${W_bh[-1]:,.2f}")

Simulación histórica completada.
  Riqueza Final DP: $256,003.96
  Riqueza Final Siempre Rebalancear: $251,091.54
  Riqueza Final Buy & Hold: $206,979.87


## 5. Visualizaciones de Resultados

Generamos los gráficos interactivos de la riqueza acumulada con marcadores para los puntos de rebalanceo de la estrategia DP, y el drawdown histórico.

In [7]:
# 1. Gráfico de evolución de Riqueza
fig_riqueza = go.Figure()
fig_riqueza.add_trace(go.Scatter(x=fechas_sim_full, y=W_dp, mode='lines', name=f'DP Optimizado (${W_dp[-1]:,.2f})', line=dict(color='green', width=3)))
fig_riqueza.add_trace(go.Scatter(x=fechas_sim_full, y=W_sr, mode='lines', name=f'Siempre Rebalancear (${W_sr[-1]:,.2f})', line=dict(color='red', dash='dash')))
fig_riqueza.add_trace(go.Scatter(x=fechas_sim_full, y=W_bh, mode='lines', name=f'Buy & Hold (${W_bh[-1]:,.2f})', line=dict(color='gray', dash='dot')))

for t_reb in rebalanceos_dp:
    fig_riqueza.add_vline(x=fechas_sim_full[t_reb+1], line_width=1.5, line_dash="dash", line_color="purple")

fig_riqueza.update_layout(title="Comparativa de Crecimiento de Capital", xaxis_title="Fecha", yaxis_title="Capital (USD)", hovermode="x unified", template='plotly_dark')
fig_riqueza.show()

# 2. Calcular y graficar Drawdowns
def calcular_drawdown(riqueza_path):
    cum_max = np.maximum.accumulate(riqueza_path)
    with np.errstate(divide='ignore', invalid='ignore'):
        dd = (riqueza_path - cum_max) / cum_max
        dd = np.nan_to_num(dd, nan=0.0)
    return dd * 100

dd_dp = calcular_drawdown(W_dp)
dd_sr = calcular_drawdown(W_sr)
dd_bh = calcular_drawdown(W_bh)

fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(x=fechas_sim_full, y=dd_dp, mode='lines', name='Drawdown DP Optimizado', line=dict(color='green')))
fig_dd.add_trace(go.Scatter(x=fechas_sim_full, y=dd_sr, mode='lines', name='Drawdown Siempre Rebalancear', line=dict(color='red', dash='dash')))
fig_dd.add_trace(go.Scatter(x=fechas_sim_full, y=dd_bh, mode='lines', name='Drawdown Buy & Hold', line=dict(color='gray', dash='dot')))

for t_reb in rebalanceos_dp:
    fig_dd.add_vline(x=fechas_sim_full[t_reb+1], line_width=1.0, line_dash="dash", line_color="purple")
    
fig_dd.update_layout(title='Caídas Máximas de las Estrategias (Drawdown)', xaxis_title='Fecha', yaxis_title='Drawdown (%)', hovermode="x unified", template='plotly_dark')
fig_dd.show()

## 6. Gráficos Descriptivos Adicionales

Incluimos la distribución del portafolio objetivo (Dona), el desempeño individual de activos (Barras) y la correlación (Heatmap) para complementar el análisis.

In [8]:
# 1. Composición del Portafolio Objetivo (Dona)
df_pesos = pd.DataFrame({'Ticker': TICKERS_ORDENADOS, 'Peso (%)': (w_optimo * 100).round(2)})
df_pesos_filtrado = df_pesos[df_pesos['Peso (%)'] > 0.01]
fig_dona = px.pie(
    df_pesos_filtrado, 
    values='Peso (%)', 
    names='Ticker', 
    title='Composición del Portafolio Objetivo (Pesos %)', 
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_dona.update_layout(template='plotly_dark')
fig_dona.show()

# 2. Desempeño Individual de los Activos
df_activos = pd.DataFrame({
    'Ticker': TICKERS_ORDENADOS,
    'Retorno Anualizado (%)': (mu * 100).round(2),
    'Volatilidad Anualizada (%)': (np.sqrt(np.diag(Sigma)) * 100).round(2)
})
df_long = df_activos.melt(id_vars='Ticker', value_vars=['Retorno Anualizado (%)', 'Volatilidad Anualizada (%)'],
                          var_name='Métrica', value_name='Valor (%)')
fig_activos = px.bar(
    df_long,
    x='Ticker',
    y='Valor (%)',
    color='Métrica',
    barmode='group',
    title='Desempeño Individual de los Activos',
    color_discrete_sequence=['#2ecc71', '#e74c3c']
)
fig_activos.update_layout(template='plotly_dark')
fig_activos.show()

# 3. Matriz de Correlación
correlaciones = retornos_diarios.corr()
fig_heatmap = px.imshow(
    correlaciones,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="Matriz de Correlación de Activos (Retornos Diarios)"
)
fig_heatmap.update_layout(template='plotly_dark')
fig_heatmap.show()

## 7. Exportación de Resultados

Guardamos el JSON de resultados en el formato establecido y exportamos la matriz de costos acumulados de Bellman a Excel.

In [9]:
metrics_m3 = {
    "parametros_dp": {
        "lambda_tc": float(LAMBDA_TC),
        "periodos_t": int(T_PERIODOS),
        "paso_grilla": float(PASO_GRILLA),
        "total_estados": int(M)
    },
    "riqueza_final": {
        "buy_and_hold": float(W_bh[-1]),
        "siempre_rebalancear": float(W_sr[-1]),
        "dp_optimizado": float(W_dp[-1])
    },
    "timeline_rebalanceos_dp": [int(x) for x in rebalanceos_dp],
    "trayectoria_dp": W_dp.tolist(),
    "fechas_completas": [d.strftime("%Y-%m-%d") for d in fechas_sim_full],
    "grilla_estados": S.tolist(),
    "matriz_politica": politica.tolist()
}

# Guardar en raíz (para app.py / páginas Streamlit)
with open("resultados_m3.json", "w", encoding="utf-8") as f:
    json.dump(metrics_m3, f, ensure_ascii=False, indent=2)

# Guardar en salidas/ (para otros notebooks)
with open("salidas/resultados_m3.json", "w", encoding="utf-8") as f:
    json.dump(metrics_m3, f, ensure_ascii=False, indent=2)

print("Archivo resultados_m3.json exportado correctamente en ambos directorios.")

# Exportar Matriz de Costos a Excel
df_export = pd.DataFrame(V)
excel_path = "salidas/matriz_dp_costos.xlsx"
df_export.to_excel(excel_path, index=False, header=False, sheet_name='Costos_DP')
print(f"Matriz de costos de Bellman exportada correctamente a {excel_path}.")

Archivo resultados_m3.json exportado correctamente en ambos directorios.


Matriz de costos de Bellman exportada correctamente a salidas/matriz_dp_costos.xlsx.
